#### Série: DuckDB Masterclass
**Ordem sugerida de leitura:** 10 de 14


📚 **Material de apoio — Laboratórios práticos no Jupyter Notebook**

Acompanhe os conceitos deste artigo com exemplos SQL, exercícios e laboratórios da DuckDB Masterclass.

🔗 [Acessar os laboratórios no GitHub](https://github.com/aestevaomoraes/duckdb-masterclass)


# DuckDB Masterclass — Capítulo 10

## Lakehouse: Parquet versus Delta e Iceberg

**Trilha:** conceito → teoria → laboratório → experimento → desafio → validação.

**Contexto:** Analytics e Engenharia de Dados, com exemplos de operações.

> Esta edição substitui os experimentos centrais por versões autocontidas. O material original está preservado integralmente na pasta `Originais/`. Quaisquer atividades dependentes de cloud/serviços externos são opcionais.

## 10.1 — Objetivos e diagnóstico inicial

**Ao final você deverá:** Criar uma pequena base Parquet particionada e reconhecer o que ela não oferece.

**Pergunta de partida:** qual decisão analítica ou de engenharia está sendo tomada neste capítulo? Registre sua hipótese antes de executar o código.

## 10.2 — Conceito e teoria

Parquet é formato de arquivo colunar; Delta Lake e Apache Iceberg acrescentam metadados de tabela e gestão de versões. Não basta um diretório de Parquet para obter transações e time travel.

### Modelo mental

- **Entrada:** que dados ou recursos estão disponíveis?
- **Operação:** o que o DuckDB transforma, consulta ou controla?
- **Saída:** que informação ou garantia pode ser verificada?
- **Limite:** o que o laboratório **não** permite concluir?

## 10.3 — Preparação reproduzível

Execute a célula a seguir. Caso falte DuckDB, instale com `pip install duckdb pandas` no ambiente virtual utilizado pelo Jupyter e reinicie o kernel. Reiniciar o kernel apaga conexões em memória.

In [ ]:
import duckdb
from IPython.display import display
print('DuckDB:', duckdb.__version__)
con = duckdb.connect(':memory:')

## 10.4 — Laboratório guiado

**Previsão:** antes de executar, descreva em uma frase o resultado esperado.

**Interpretação:** depois de executar, explique o que cada coluna ou efeito representa.

In [ ]:
con.execute("CREATE TABLE vendas AS SELECT i id, CASE WHEN i%2=0 THEN 'MS' ELSE 'SP' END uf, (i%20+1) AS valor FROM range(40) t(i)")
con.execute("COPY vendas TO 'lake_parquet' (FORMAT PARQUET, PARTITION_BY (uf), OVERWRITE_OR_IGNORE TRUE)")
res=con.execute("SELECT uf, SUM(valor) AS total FROM read_parquet('lake_parquet/**/*.parquet', hive_partitioning=true) GROUP BY ALL ORDER BY uf").df()
display(res)
assert res['total'].sum()==con.execute('SELECT SUM(valor) FROM vendas').fetchone()[0]

## 10.5 — Experimentação controlada

Altere um parâmetro de cada vez. Compare a saída com a hipótese e registre se a mudança alterou os dados, o plano ou apenas a apresentação.

In [ ]:
display(con.execute("SELECT DISTINCT uf FROM read_parquet('lake_parquet/**/*.parquet', hive_partitioning=true) ORDER BY uf").df())

## 10.6 — Desafio profissional

Uma equipe precisa de time travel e atualizações concorrentes: por que particionar Parquet não resolve sozinho?

**Critérios de aceite:** descreva a regra de negócio, escreva uma consulta/experimento, valide o resultado e explique pelo menos uma limitação.

**Solução:** registre sua resposta na célula abaixo antes de consultar o material original.

In [ ]:
# Sua solução: escreva aqui seu experimento ou explicação.


## 10.7 — Autoavaliação e visão de especialista

1. Explique o conceito sem consultar o código.
2. Identifique a hipótese principal do laboratório.
3. Cite um erro de dados ou de configuração que poderia invalidar o resultado.
4. Explique como você verificaria esse erro.
5. Relacione a técnica a um caso real de Operations Analytics.

**Próximo passo da mentoria:** revisar suas respostas e os resultados de execução antes de avançar.

## 10.8 — Leituras e materiais originais

- [Documentação oficial DuckDB](https://duckdb.org/docs/stable/) — consulte o tópico específico antes de usar recursos em produção.
- Consulte o notebook de mesmo nome em `Originais/` para exemplos históricos e contexto adicional.

**Estado de validação:** estrutura e sintaxe Python verificadas no empacotamento; consultas SQL requerem execução no ambiente com DuckDB.